# Introduction to Arabic Speech Technologies
## Chapter 2 notebook: Building a small Arabic pronunciation lexicon

Electronic supplementary material for *Introduction to Arabic Speech Technologies* by Hend S. Al-Khalifa.

Chapter 2 introduces the sounds of Arabic, the writing system and the diacritics, and notes that classical recognisers (Chapter 4) and synthesisers (Chapter 9) both depend on a lexicon that maps written words to phone sequences. This notebook builds such a lexicon from diacritized words, with the rules the chapter describes: short and long vowels, gemination marked by shadda, the sun and moon behaviour of the definite article, tāʾ marbūṭa in pause and in context, and nunation.

**Contents**

1. Letters, diacritics and a phone inventory
2. A grapheme-to-phoneme function
3. The definite article: sun and moon letters
4. Gemination and tāʾ marbūṭa
5. Building and writing the lexicon
6. Undiacritized input: why one spelling gives several entries

**Running it.** The notebook needs only `numpy`, `scipy` and `matplotlib` (see `requirements.txt`). It runs top to bottom with no downloads and no audio files: where a recording is useful, the notebook synthesises one, and a cell is provided for reading your own WAV file instead. Optional cells that need extra packages or internet access are marked *Optional*.


## 1. Letters, diacritics and a phone inventory

Phones are written in IPA. Emphatic consonants carry the pharyngealisation mark, long vowels the length mark.

In [1]:
CONSONANTS = {
 "\u0621":"\u0294","\u0628":"b","\u062A":"t","\u062B":"\u03b8","\u062C":"d\u0292","\u062D":"\u0127","\u062E":"x",
 "\u062F":"d","\u0630":"\u00f0","\u0631":"r","\u0632":"z","\u0633":"s","\u0634":"\u0283","\u0635":"s\u02e4",
 "\u0636":"d\u02e4","\u0637":"t\u02e4","\u0638":"\u00f0\u02e4","\u0639":"\u0295","\u063A":"\u0263","\u0641":"f",
 "\u0642":"q","\u0643":"k","\u0644":"l","\u0645":"m","\u0646":"n","\u0647":"h","\u0648":"w","\u064A":"j",
}
SHORT = {"\u064E":"a", "\u0650":"i", "\u064F":"u"}          # fatha, kasra, damma
TANWIN = {"\u064B":"an", "\u064D":"in", "\u064C":"un"}      # nunation
SHADDA, SUKUN = "\u0651", "\u0652"
SUN_LETTERS = set("\u062A\u062B\u062F\u0630\u0631\u0632\u0633\u0634\u0635\u0636\u0637\u0638\u0644\u0646")

print("consonant phones:", len(CONSONANTS))
print("emphatics:", [f"{k} -> {v}" for k, v in CONSONANTS.items() if "\u02e4" in v])

consonant phones: 28
emphatics: ['ص -> sˤ', 'ض -> dˤ', 'ط -> tˤ', 'ظ -> ðˤ']


## 2. Grapheme to phoneme

The function walks the diacritized string. A consonant followed by shadda is doubled; a long vowel is a short vowel of the same quality followed by its matching glide letter (ا, و, ي); sukūn means no vowel follows.

In [2]:
def canonical(word):
    # Some sources write the vowel before the shadda; put the shadda next to its consonant.
    out = []
    for ch in word:
        if ch == SHADDA and out and out[-1] in SHORT:
            vowel = out.pop(); out.append(SHADDA); out.append(vowel)
        else:
            out.append(ch)
    return "".join(out)

def g2p(word):
    out, i = [], 0
    chars = list(canonical(word))
    while i < len(chars):
        ch = chars[i]
        if ch in CONSONANTS:
            phone = CONSONANTS[ch]
            nxt = chars[i + 1] if i + 1 < len(chars) else ""
            if nxt == SHADDA:
                out.append(phone); out.append(phone)      # geminate: written once, said long
                i += 2
                continue
            out.append(phone); i += 1
            continue
        if ch in SHORT:
            vowel = SHORT[ch]
            nxt = chars[i + 1] if i + 1 < len(chars) else ""
            long_pairs = {("a", "\u0627"), ("u", "\u0648"), ("i", "\u064A")}
            if (vowel, nxt) in long_pairs:
                out.append(vowel + "\u02D0"); i += 2                  # long vowel
            else:
                out.append(vowel); i += 1
            continue
        if ch in TANWIN:
            out.extend(list(TANWIN[ch])); i += 1; continue
        if ch == SUKUN:
            i += 1; continue
        if ch == "\u0627":                                   # bare alif, e.g. after a consonant with no mark
            out.append("a\u02D0"); i += 1; continue
        if ch == "\u0629":                                   # ta marbuta: silent in pause after fatha
            if not out or out[-1] != "a":
                out.append("a")
            i += 1; continue
        if ch == "\u0649":
            out.append("a\u02D0"); i += 1; continue
        i += 1
    return out

for w in ["\u0643\u064E\u062A\u064E\u0628\u064E", "\u0643\u064E\u062A\u0651\u064E\u0628\u064E", "\u0628\u064E\u0627\u0628", "\u062A\u0650\u064A\u0646", "\u0637\u0650\u064A\u0646"]:
    print(f"{w:12} -> /{' '.join(g2p(w))}/")

كَتَبَ       -> /k a t a b a/
كَتَّبَ      -> /k a t t a b a/
بَاب         -> /b aː b/
تِين         -> /t iː n/
طِين         -> /tˤ iː n/


The third and fourth lines are the minimal pair of Chapter 2: تِين *tīn* and طِين *ṭīn* differ only in the emphatic quality of the first consonant, which the transcription marks with ˤ and which colours the following vowel.

## 3. The definite article

Before a sun letter the /l/ of الـ assimilates and the following consonant is pronounced long, so الشَّمْس is *ash-shams*, not *al-shams*. Before a moon letter the /l/ is kept.

In [3]:
def apply_article(word):
    """Return (stem, is_definite, assimilated) for a word written with the definite article."""
    if word.startswith("\u0627\u0644") and len(word) > 2:
        rest = word[2:]
        first = next((c for c in rest if c in CONSONANTS), "")
        return rest, True, first in SUN_LETTERS
    return word, False, False

def g2p_word(word):
    stem, definite, assimilated = apply_article(word)
    phones = g2p(stem)
    if definite:
        if assimilated and phones:
            doubled_in_text = len(phones) > 1 and phones[0] == phones[1]
            head = phones if doubled_in_text else [phones[0]] + phones
            return ["\u0294", "a"] + head                      # the /l/ assimilates, the sun letter is long
        return ["\u0294", "a", "l"] + phones
    return phones

for w in ["\u0627\u0644\u0634\u0651\u064E\u0645\u0652\u0633", "\u0627\u0644\u0642\u064E\u0645\u064E\u0631", "\u0627\u0644\u0645\u064E\u062F\u0652\u0631\u064E\u0633\u064E\u0629"]:
    stem, definite, assim = apply_article(w)
    kind = "sun (assimilated)" if assim else "moon"
    print(f"{w:14} {kind:20} -> /{' '.join(g2p_word(w))}/")

الشَّمْس       sun (assimilated)    -> /ʔ a ʃ ʃ a m s/
القَمَر        moon                 -> /ʔ a l q a m a r/
المَدْرَسَة    moon                 -> /ʔ a l m a d r a s a/


## 4. Gemination and tāʾ marbūṭa

Gemination is contrastive and morphologically productive: doubling the middle consonant of a root derives a Form II verb. Tāʾ marbūṭa is pronounced [a] in pause but /t/ before a suffix or in a construct phrase, so the lexicon needs both forms.

In [4]:
pairs = [("\u062F\u064E\u0631\u064E\u0633\u064E", "\u062F\u064E\u0631\u064E\u0651\u0633\u064E", "he studied / he taught"),
         ("\u0643\u064E\u0633\u064E\u0631\u064E", "\u0643\u064E\u0633\u064E\u0651\u0631\u064E", "he broke / he smashed")]
for form1, form2, gloss in pairs:
    print(f"{form1} /{' '.join(g2p(form1))}/   vs   {form2} /{' '.join(g2p(form2))}/   ({gloss})")

def ta_marbuta_variants(word):
    if word.endswith("\u0629"):
        pause = g2p_word(word)
        context = g2p_word(word[:-1] + "\u062A") + ["u"]     # ...atu in context
        return {"pause": pause, "context": context}
    return {"pause": g2p_word(word)}

madrasa = "\u0627\u0644\u0645\u064E\u062F\u0652\u0631\u064E\u0633\u064E\u0629"
for form, phones in ta_marbuta_variants(madrasa).items():
    print(f"{madrasa} ({form:7}) -> /{' '.join(phones)}/")

دَرَسَ /d a r a s a/   vs   دَرَّسَ /d a r r a s a/   (he studied / he taught)
كَسَرَ /k a s a r a/   vs   كَسَّرَ /k a s s a r a/   (he broke / he smashed)
المَدْرَسَة (pause  ) -> /ʔ a l m a d r a s a/
المَدْرَسَة (context) -> /ʔ a l m a d r a s a t u/


## 5. Building the lexicon

A pronunciation lexicon lists each written form with one or more phone sequences. Alternatives matter: a recogniser that allows only one pronunciation per word will penalise a speaker who uses another.

In [5]:
WORDLIST = ["\u0643\u064E\u062A\u064E\u0628\u064E", "\u0643\u064E\u062A\u0651\u064E\u0628\u064E", "\u0645\u064E\u062F\u0652\u0631\u064E\u0633\u064E\u0629",
            "\u0627\u0644\u0645\u064E\u062F\u0652\u0631\u064E\u0633\u064E\u0629", "\u0627\u0644\u0634\u0651\u064E\u0645\u0652\u0633", "\u0627\u0644\u0642\u064E\u0645\u064E\u0631",
            "\u062A\u0650\u064A\u0646", "\u0637\u0650\u064A\u0646", "\u0628\u064E\u0627\u0628", "\u0643\u0650\u062A\u064E\u0627\u0628"]

lexicon = {}
for w in WORDLIST:
    variants = ta_marbuta_variants(w)
    lexicon[w] = [" ".join(p) for p in variants.values()]

for w, prons in lexicon.items():
    for p in prons:
        print(f"{w}\t{p}")

with open("arabic_lexicon.txt", "w", encoding="utf-8") as f:
    for w, prons in lexicon.items():
        for p in prons:
            f.write(f"{w}\t{p}\n")
print(f"\n{sum(len(v) for v in lexicon.values())} entries written to arabic_lexicon.txt")

كَتَبَ	k a t a b a
كَتَّبَ	k a t t a b a
مَدْرَسَة	m a d r a s a
مَدْرَسَة	m a d r a s a t u
المَدْرَسَة	ʔ a l m a d r a s a
المَدْرَسَة	ʔ a l m a d r a s a t u
الشَّمْس	ʔ a ʃ ʃ a m s
القَمَر	ʔ a l q a m a r
تِين	t iː n
طِين	tˤ iː n
بَاب	b aː b
كِتَاب	k i t aː b

12 entries written to arabic_lexicon.txt


## 6. Undiacritized input

Ordinary Arabic text leaves the short vowels unwritten, so one spelling can stand for several words. A lexicon built from undiacritized text therefore needs one entry per reading, and a synthesiser must choose between them from context. This is the ambiguity Chapter 9 returns to.

In [6]:
readings = {
 "\u0639\u0644\u0645": [("\u0639\u0650\u0644\u0652\u0645", "knowledge"), ("\u0639\u064E\u0644\u064E\u0645", "flag"),
          ("\u0639\u064E\u0644\u0650\u0645\u064E", "he knew"), ("\u0639\u064E\u0644\u064E\u0651\u0645\u064E", "he taught")],
}
for bare, options in readings.items():
    print(f"written form: {bare}")
    for vocalised, gloss in options:
        print(f"   {vocalised:10} /{' '.join(g2p(vocalised)):22}/  {gloss}")

written form: علم
   عِلْم      /ʕ i l m               /  knowledge
   عَلَم      /ʕ a l a m             /  flag
   عَلِمَ     /ʕ a l i m a           /  he knew
   عَلَّمَ    /ʕ a l l a m a         /  he taught
